# Cohort Creation Workflow

This notebook provides a complete workflow for clearing and rebuilding cohorts.

## Workflow Steps

1. **Configuration**: Set up paths, Python binaries, and cohort parameters
2. **Cleanup**: Clear existing cohort data (S3 and local)
3. **Cohort Creation**: Build cohorts with fall injury and ED event logic
4. **Verification**: Verify cohorts were created successfully

## Navigation

- **Section A**: Configuration and Setup
- **Section B**: Cleanup Existing Cohorts
- **Section C**: Create Cohorts
- **Section D**: Verify Cohort Creation

---

**Cohorts:**
- `falls` — `fall_injury_any = 1` (injury S00–S99 + external cause W00–W19 for the same patient within `CPIC_FALL_TARGET_WINDOW_DAYS`, default 7 days)
- `ed` — `ed_event = 1` (POS=23 or revenue code 045x/0981)

## A. Configuration and Setup

In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import boto3
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set project root
PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd()
if PROJECT_ROOT.name == "2_create_cohort":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PYTHON_BIN = Path(sys.executable)

from py_helpers.env_utils import get_data_root

# Age bands: 65–85 group only (65-74, 75-84)
_age_bands = ["65-74", "75-84"]

try:
    from py_helpers.constants import REQUIRED_COHORTS
    COHORTS = list(REQUIRED_COHORTS.keys())  # ["falls", "ed"]
    AGE_BANDS = {k: [b for b in v if b in _age_bands] for k, v in REQUIRED_COHORTS.items()}
except ImportError:
    COHORTS = ["falls", "ed"]
    AGE_BANDS = {"falls": _age_bands, "ed": _age_bands}

EVENT_YEARS = [2016, 2017, 2018, 2019]
S3_BUCKET = os.environ.get("CPIC_S3_BUCKET", "pgxdatalake")

DATA_ROOT = get_data_root()
duckdb_tmp = Path("/mnt/nvme/duckdb_tmp") if Path("/mnt/nvme").exists() else Path.home() / "duckdb_tmp"
os.environ["DUCKDB_TMP_DIRECTORY"] = str(duckdb_tmp)
os.environ["CPIC_TOTAL_WORKERS"] = "2"

LOG_DIR = PROJECT_ROOT / "2_create_cohort" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Python Binary: {PYTHON_BIN}")
print(f"Cohorts: {COHORTS}")
print(f"Age bands (65–85 group only): {AGE_BANDS}")
print(f"Event Years: {EVENT_YEARS}")
print(f"Data Root: {DATA_ROOT}")
print(f"Log Directory: {LOG_DIR}")

## B. Cleanup Existing Cohorts

**Warning**: This will delete existing cohort data from S3 and local storage.

This step clears:
- Step 2: Cohort parquet files (S3 and local)
- Step 3b: Feature importance outputs
- Step 4a: Model data
- Step 6: Trained models
- Checkpoints (optional)

In [ ]:
# Check what exists before cleanup
s3_client = boto3.client('s3')

print("📊 Checking existing cohort data in S3...")
print("=" * 60)

cohort_paths = [
    f"gold/cohorts/cohort_name={cohort}/" for cohort in COHORTS
]

for cohort_path in cohort_paths:
    try:
        response = s3_client.list_objects_v2(
            Bucket=S3_BUCKET,
            Prefix=cohort_path,
            MaxKeys=10
        )
        
        if 'Contents' in response:
            count = len(response['Contents'])
            total = response.get('KeyCount', count)
            print(f"   {cohort_path}: {count} objects found (showing first 10)")
        else:
            print(f"   {cohort_path}: No objects found")
    except Exception as e:
        print(f"   {cohort_path}: Error checking - {e}")

print("=" * 60)

In [ ]:
# Run cleanup script
# Options:
#   --skip-checkpoints: Keep checkpoints
#   --skip-s3: Only delete local files
#   --skip-local: Only delete S3 files
#   --yes: Auto-confirm (skip interactive prompt)

SKIP_CHECKPOINTS = False  # Set to True to keep checkpoints
SKIP_S3 = False           # Set to True to skip S3 deletion
SKIP_LOCAL = False        # Set to True to skip local deletion
AUTO_CONFIRM = True       # Set to True to skip confirmation prompt (for notebook use)

cleanup_script = PROJECT_ROOT / "utility_scripts" / "cleanup_cohort_data.sh"

if not cleanup_script.exists():
    print(f"❌ Cleanup script not found: {cleanup_script}")
    print("   Skipping cleanup step")
else:
    print("🧹 Running cleanup script...")
    print(f"   Script: {cleanup_script}")
    
    # Build command
    cmd = ["bash", str(cleanup_script)]
    
    if SKIP_CHECKPOINTS:
        cmd.append("--skip-checkpoints")
    if SKIP_S3:
        cmd.append("--skip-s3")
    if SKIP_LOCAL:
        cmd.append("--skip-local")
    if AUTO_CONFIRM:
        cmd.append("--yes")  # Skip confirmation prompt for notebook use
    
    print(f"   Command: {' '.join(cmd)}")
    if AUTO_CONFIRM:
        print(f"   ⚠️  Auto-confirmation enabled (--yes flag)")
    
    result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True)
    
    if result.returncode == 0:
        print(f"\n✅ Cleanup completed successfully")
    else:
        print(f"\n❌ Cleanup failed with return code {result.returncode}")
        if result.stdout:
            print(f"   Output: {result.stdout[-500:]}")  # Last 500 chars
        if result.stderr:
            print(f"   Error: {result.stderr[-500:]}")  # Last 500 chars

## C. Create Cohorts

Two cohort series:

1. **FALLS** (`falls`): `fall_injury_any = 1` — injury diagnosis (S00–S99/T07/T14/T20–T34/T79) AND external cause W00–W19 for the same patient within `CPIC_FALL_TARGET_WINDOW_DAYS` (default 7 days).
2. **ED** (`ed`): `ed_event = 1` — place of service = 23 (Emergency Room) or revenue code 045x/0981.

### Falls Cohort

In [ ]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_falls.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
print(f"Exit code: {result.returncode}")

### ED Cohort

In [ ]:
import subprocess

script = PROJECT_ROOT / "2_create_cohort" / "run_series_ed.py"
cmd = [str(PYTHON_BIN), str(script), "--skip-existing", "--concurrent-workers", "1"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
print(f"Exit code: {result.returncode}")

## D. Verify Cohort Creation

This section verifies that cohorts were created successfully by checking S3 and local paths.

In [ ]:
import boto3

s3_client = boto3.client('s3')

print("Verifying cohorts in S3...")
print("=" * 60)

all_found = True
for cohort in COHORTS:
    for age_band in AGE_BANDS.get(cohort, []):
        for event_year in EVENT_YEARS:
            s3_key = f"gold/cohorts/cohort_name={cohort}/event_year={event_year}/age_band={age_band}/cohort.parquet"
            try:
                response = s3_client.head_object(Bucket=S3_BUCKET, Key=s3_key)
                size_mb = response['ContentLength'] / (1024 * 1024)
                print(f"  OK  {cohort}/{age_band}/{event_year}: {size_mb:.2f} MB")
            except s3_client.exceptions.ClientError as e:
                code = e.response['Error']['Code']
                if code == '404':
                    print(f"  MISSING  {cohort}/{age_band}/{event_year}")
                    all_found = False
                else:
                    print(f"  ERROR  {cohort}/{age_band}/{event_year}: {e}")
                    all_found = False

print("=" * 60)
print("All cohorts verified." if all_found else "Some cohorts missing — check logs.")

In [ ]:
import duckdb
from py_helpers.env_utils import get_data_root

data_root = get_data_root()
local_cohorts = data_root / "gold" / "cohorts"

print("Verifying target columns for falls cohort...")
print("=" * 60)

test_cohort = "falls"
test_age_band = AGE_BANDS.get("falls", ["65-74"])[0]
test_year = EVENT_YEARS[0]

local_path = local_cohorts / f"cohort_name={test_cohort}" / f"event_year={test_year}" / f"age_band={test_age_band}" / "cohort.parquet"
print(f"Checking: {local_path}")

if not local_path.exists():
    print(f"File not found locally — sync from S3 first.")
else:
    con = duckdb.connect()
    try:
        schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{local_path}') LIMIT 0").fetchdf()
        cols = set(schema['column_name'].tolist())
        expected = ["fall_injury_any", "fall_injury_serious", "fall_injury_head", "ed_event"]
        for col in expected:
            status = "OK" if col in cols else "MISSING"
            print(f"  {status}  {col}")
        feature_flags = ["r29_6_flag", "z91_81_flag", "cpt_1100f_flag"]
        for col in feature_flags:
            status = "OK" if col in cols else "not present (check event filter)"
            print(f"  {status}  {col}  (feature flag)")
    finally:
        con.close()

## Next Steps

After successfully creating cohorts, proceed to:

1. **Step 3b: Feature Importance EDA**
   - Run BupaR post-target analysis
   - Filter post-target leakage features
   - Generate refined feature importance files

2. **Step 4a: Model Data Creation**
   - Create `model_events.parquet` with cases and controls
   - Uses refined features from Step 3b

3. **Step 4b: Event Filtering**
   - Filter administrative codes
   - Remove post-event leakage

4. **Step 5: PGx Feature Engineering**
   - Add PGx drug counts
   - Add drug counts

5. **Step 6: Final Model Training**
   - Train CatBoost, XGBoost, and XGBoost RF models
   - Select best model by recall/AUC-PR

See `WORKFLOW_EXECUTION_TODO.md` for detailed instructions.